In [1]:
import glob
import os 
import re
import sys
import copy
import itertools
import random
import numpy as np
import pandas as pd
from PIL import Image
import json

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression, Lasso, LassoCV, Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torchvision import models
from torchvision.models import resnet18, ResNet18_Weights

In [4]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW

/Users/amamotoyoshifumi/miniconda3/envs/pytorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger

In [6]:
import sklearn, transformers
print ("numpy: ", np.__version__)
print ("pandas: ", pd.__version__)
print ("sklearn: ", sklearn.__version__)
print ("transformers: ", transformers.__version__)

numpy:  2.4.3
pandas:  3.0.2
sklearn:  1.8.0
transformers:  5.8.0


In [7]:
path="./"

In [ ]:
seed = 0  # 0-9


In [ ]:
latent_list=[2,4,8,16,32,64,128,256,512]
kind_list=["all", "IR", "WAXS", "POM"]

In [ ]:
latent_model = "AE1"     # one of: AE1, VAE1, CL1, CL3, PG1


In [ ]:
max_epoch=500

# 1. Loading Data

In [ ]:
PROPERTY_NAMES = [
    "degradation",
    "elongation_break",
    "young_modulus",
    "crystallinity"
]

In [8]:
df_param=pd.read_csv(path+'/data/params/param_sum.csv', index_col=0)
df_dc=df_param[['WAXS_Int1','WAXS_Int2','WAXS_Int3','WAXS_Int4','WAXS_Int5','WAXS_Int6','WAXS_Int7','WAXS_Int8','WAXS_Int9']].sum(axis=1)/df_param[['WAXS_Int1','WAXS_Int2','WAXS_Int3','WAXS_Int4','WAXS_Int5','WAXS_Int6','WAXS_Int7','WAXS_Int8','WAXS_Int9','WAXS_Int10','WAXS_Int11']].sum(axis=1)
df_param=pd.read_csv(path+'/data/params/param_sum.csv',index_col=0)
df_pred=pd.concat([df_param[['Property_Deg', 'Property_StrainBreak', 'Property_Young']], df_dc], axis=1)
df_pred['Property_Deg']=df_pred['Property_Deg']*100
df_pred['Property_StrainBreak']=df_pred['Property_StrainBreak']*100
df_pred['Property_Young']=df_pred['Property_Young']/1000
df_pred.columns=["Deg","Strain","Young","CrysDeg"]
df_pred["CrysDeg"]=df_pred["CrysDeg"]*100
df_pred

,Deg,Strain,Young,CrysDeg
17_1,7.8653,15.9800,1.533741,4.297316
18_1,8.6981,8.8500,1.595907,4.929744
19_1,8.1722,5.7200,1.704735,12.826432
21_1,7.8401,7.9833,1.549706,2.253836
22_1,8.7794,17.7067,1.499181,2.424432
...,...,...,...,...
78_3,0.9879,3.4867,1.853111,26.101993
79_3,1.0791,3.9867,1.812193,59.524230
81_3,8.0662,11.6667,1.587941,2.561380
82_3,7.8363,9.9367,1.442483,2.412784


# 2. Q&A LLM Using Latent Embeddings

In [14]:
class MaterialDataset(Dataset):

    def __init__(
        self,
        latent_array,
        y_df,
        qa_jsonl_dir,
        sample_list,
        tokenizer,
        max_len=64,
        mode="train",
        test_qa_idx=0   
    ):

        self.latent = torch.tensor(latent_array, dtype=torch.float32)

        self.y = torch.tensor(y_df.values, dtype=torch.float32)

        self.sample_list = sample_list
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.mode = mode

        self.epoch = 0

        self.test_qa_idx = test_qa_idx

        self.qa_data = []

        for i in range(len(self.sample_list)):

            path = qa_jsonl_dir + '/' + self.sample_list[i] + '.jsonl'

            with open(path, "r", encoding="utf-8") as f:
                qa_list = [json.loads(line) for line in f]

            self.qa_data.append(qa_list)

    def __len__(self):
        return len(self.latent)

    def set_epoch(self, epoch):
        self.epoch = epoch

    def set_test_qa_idx(self, idx):
        self.test_qa_idx = idx

    def __getitem__(self, idx):
    
        latent = self.latent[idx]
        y = self.y[idx]
    
        qa_list = self.qa_data[idx]
    
        # =========================
        # train
        # =========================
        if self.mode == "train":
    
            seed = self.epoch * 1000 + idx
            rng = random.Random(seed)
    
            qa = rng.choice(qa_list)
    
        # =========================
        # test
        # =========================
        else:
    
            qa_idx = min(self.test_qa_idx, len(qa_list)-1)
    
            qa = qa_list[qa_idx]
    
        question = qa["question"]
        answer = qa["answer"]
    
        # -----------------------------
        # Prompt 
        # -----------------------------
        prompt_text = f"Question:\n{question}\n\nAnswer:\n"
    
        # -----------------------------
        # Full text
        # -----------------------------
        full_text = prompt_text + answer + self.tokenizer.eos_token
    
        # -----------------------------
        # tokenize full text
        # -----------------------------
        enc = self.tokenizer(
            full_text,
            add_special_tokens=False,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
    
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
    
        # -----------------------------
        # prompt length
        # -----------------------------
        prompt_ids = self.tokenizer(
            prompt_text,
            add_special_tokens=False
        )["input_ids"]
    
        prompt_len = len(prompt_ids)
    
        # -----------------------------
        # labels
        # -----------------------------
        labels = input_ids.clone()
    
        labels[:prompt_len] = -100
    
        labels[attention_mask == 0] = -100
    
        return {
            "latent": latent,
            "y": y,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "question": question,
            "answer": answer
        }

In [15]:
class MultiTaskGPT2(nn.Module):

    def __init__(self, latent_dim=144, hidden=512, alpha=0.1, num_prefix=8):
        super().__init__()

        self.gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")
        for p in self.gpt2.parameters():
            p.requires_grad = False
        self.embed = self.gpt2.transformer.wte

        # latent → prefix token
        self.prompt_proj = nn.Sequential(
            nn.Linear(latent_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, num_prefix * self.gpt2.config.n_embd)
        )

        self.alpha = alpha
        self.gpt2.config.pad_token_id = self.gpt2.config.eos_token_id
        self.num_prefix = num_prefix

    def extract_numbers(self, text):
        nums = re.findall(r"[-+]?\d*\.\d+|\d+", text)
        return [float(n) for n in nums]

    # =========================
    # forward
    # =========================
    # NOTE: the original signature was (..., labels, y=None). The `y` argument
    # only fed a dead block that ran a generation and discarded it, so it was
    # removed here. The property values reach the pipeline through the
    # generated text, parsed by regex -- there is no multi-task head.
    def forward(self, latent, input_ids, attention_mask, labels):

        B = latent.size(0)

        token_embeds = self.embed(input_ids)

        prompt = self.prompt_proj(latent)

        prompt = prompt.view(B, self.num_prefix, self.gpt2.config.n_embd)

        inputs_embeds = torch.cat([prompt, token_embeds], dim=1)

        prompt_mask = torch.ones(B, self.num_prefix, device=attention_mask.device)
        attention_mask = torch.cat([prompt_mask, attention_mask], dim=1)

        labels = torch.cat([torch.full((B, self.num_prefix), -100, device=latent.device, dtype=torch.long), labels], dim=1)

        out = self.gpt2(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=True,
            return_dict=True
        )

        loss_text = out.loss

        return loss_text

In [16]:
import re

def extract_material_properties(text):

    patterns = {
        "crystallinity": r"(?:Crystallinity degree|crystallinity degree).*?(?:is|of|at|amounts to|=)?\s*([0-9]+(?:\.[0-9]+)?)\s*%?",
        "degradation": r"(?:Degradation rate|degradation rate).*?(?:is|of|at|=)?\s*([0-9]+(?:\.[0-9]+)?)\s*%?",
        "strain": r"(?:Strain at break|strain at break).*?(?:is|of|at|amounts to|=)?\s*([0-9]+(?:\.[0-9]+)?)\s*%?",
        "young_modulus": r"(?:Young's modulus|young's modulus).*?(?:is|of|at|=)?\s*([0-9]+(?:\.[0-9]+)?)\s*GPa?",
    }

    results = {}

    for key, pattern in patterns.items():
        matches = re.findall(pattern, text, flags=re.IGNORECASE)

        if matches:
            valid = []
            for m in matches:
                try:
                    val = float(m)
                    valid.append(val)
                except:
                    continue

            if valid:
                results[key] = valid[-1] 

    return results

In [ ]:
for kind in kind_list:
    for latent in latent_list:
        # The original strings started with "/", so the real path became
        # "./model//Analysis7/...". The leading slash is removed here.
        model_path = "Analysis7/"+latent_model+"/Analysis7-"+latent_model+'_'+str(latent)+'_'+str(seed)+'_'+kind+".pth"
        result_path = "Analysis7/"+latent_model+"/Analysis7-"+latent_model+'_'+str(latent)+'_'+str(seed)+'_'+kind

        # Neither directory was ever created in the original, so a 500-epoch
        # run was discarded by a failing torch.save at the very end.
        os.makedirs(os.path.dirname(path+"/model/"+model_path), exist_ok=True)
        os.makedirs(os.path.dirname(path+"/result/"+result_path), exist_ok=True)

        
        mid_train=pd.read_csv(path+'/data/mid/'+latent_model+'/mid_train'+'_'+latent_model+'_'+str(latent)+'_'+str(seed)+'.csv', ).iloc[:,1:]
        mid_test =pd.read_csv(path+'/data/mid/'+latent_model+'/mid_test'+'_'+latent_model+'_'+str(latent)+'_'+str(seed)+'.csv', ).iloc[:,1:]
        eps = np.finfo(np.float64).eps
        ave=copy.copy(np.repeat([mid_train.iloc[:,:latent].mean().mean()+eps,mid_train.iloc[:,latent:latent*2].mean().mean()+eps,mid_train.iloc[:,latent*2:].mean().mean()+eps], latent))
        mid_train=mid_train/ave
        mid_test=mid_test/ave
        y_train=pd.read_csv(path+'/data/mid/'+latent_model+'/y_train'+'_'+latent_model+'_'+str(latent)+'_'+str(seed)+'.csv', ).iloc[:,1:]
        y_test =pd.read_csv(path+'/data/mid/'+latent_model+'/y_test'+'_'+latent_model+'_'+str(latent)+'_'+str(seed)+'.csv', ).iloc[:,1:]
        
        if kind=="all":
            mid_train, mid_test=mid_train.values, mid_test.values
        elif kind=="IR":
            mid_train, mid_test=mid_train.iloc[:,:latent].values, mid_test.iloc[:,:latent].values
        elif kind=="WAXS":
            mid_train, mid_test=mid_train.iloc[:,latent:latent*2].values, mid_test.iloc[:,latent:latent*2].values
        elif kind=="POM":
            mid_train, mid_test=mid_train.iloc[:,latent*2:].values, mid_test.iloc[:,latent*2:].values
        
        y_train.columns=df_pred.columns
        y_train=y_train*df_pred.std()+df_pred.mean()
        y_train.columns=PROPERTY_NAMES
        y_test.columns=df_pred.columns
        y_test=y_test*df_pred.std()+df_pred.mean()
        y_test.columns=PROPERTY_NAMES

        tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        tokenizer.pad_token = tokenizer.eos_token
        
        train_groups, test_groups = train_test_split([i for i in range(51)], test_size=0.2, random_state=seed)
        train_index=train_groups+[i+51 for i in train_groups]+[i+102 for i in train_groups]
        test_index=test_groups+[i+51 for i in test_groups]+[i+102 for i in test_groups]
        
        qa_jsonl_dir=path+"/data/Q&A/treated"
        train_dataset = MaterialDataset(mid_train, y_train, qa_jsonl_dir, df_pred.index[train_index], tokenizer)
        test_dataset  = MaterialDataset(mid_test, y_test, qa_jsonl_dir, df_pred.index[test_index], tokenizer,
                                       mode="test", )
        
        train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

        if torch.cuda.is_available():
            device = torch.device("cuda")
        elif torch.backends.mps.is_available():
            device = torch.device("mps")
        else:
            device = torch.device("cpu")
               
        model = MultiTaskGPT2(latent_dim=mid_train.shape[1]).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
        
        # =========================
        # 1. Load an existing model
        # =========================
        if os.path.exists(path+'/model/'+model_path):
            print("Loading saved model ...")
        
            model.load_state_dict(torch.load(path+'/model/'+model_path, map_location=device))
            model.eval()
            print("Loaded saved model ...")
        
        else:
            print("No saved model found. Training from scratch.")
        
            # =========================
            # 2. Train
            # =========================
        
            for epoch in range(max_epoch):
                train_loader.dataset.set_epoch(epoch)
                model.train()
                total_loss = 0
            
                for batch in train_loader:
            
                    latent = batch["latent"].to(device)
                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    labels=batch["labels"].to(device)
                    #y = batch["y"].to(device)
                    #print (batch["question"])
                    
                    optimizer.zero_grad()
                    loss = model(latent, input_ids, attention_mask, labels)
                    loss.backward()
                    optimizer.step()
            
                    total_loss += loss.item()
            
                print(f"Epoch {epoch+1} | loss = {total_loss/len(train_loader):.4f}")
        
            # =========================
            # 3. Save
            # =========================
            torch.save(model.state_dict(), path+'/model/'+model_path)
            print("Model saved.")

        if os.path.exists(path+"/result/"+result_path+"_true.json") and os.path.exists(path+"/result/"+result_path+"_pred.json"):
            with open(path+"/result/"+result_path+"_true.json", "r", encoding="utf-8") as f:
                result_true_all = json.load(f)    
            with open(path+"/result/"+result_path+"_pred.json", "r", encoding="utf-8") as f:
                result_pred_all = json.load(f)
        
        else:
            model = MultiTaskGPT2(latent_dim=mid_train.shape[1]).to(device)
            
            state_dict = torch.load(path + '/model/' + model_path, map_location=torch.device('cpu'))
            model.load_state_dict(state_dict)
            model.to(device)
            model.eval()
            
            result_true_all, result_pred_all=[], []
            for Sample_No in range(len(test_dataset)):
                #result_true, result_pred=[], []
                if Sample_No%5==0:
                    print ("*===== Sample No "+str(Sample_No)+" =====*")
                samples=[]
                for qa_idx in range(100):
                    test_dataset.set_test_qa_idx(qa_idx)
                    samples.append(test_dataset[Sample_No])
        
                # -----------------------------
                # latent
                # -----------------------------
                latent = samples[0]["latent"].unsqueeze(0)
                latent = latent.repeat(len(samples), 1).to(device)
        
                # -----------------------------
                # question / answer
                # -----------------------------
                questions = [s["question"] for s in samples]
                answers = [s["answer"] for s in samples]
        
                prompt_texts = [
                    f"Question:\n{q}\n\nAnswer:\n"
                    for q in questions
                ]
        
                # ==========================================
                # Tokenize（100 Q&A Pairs）
                # ==========================================
                enc = tokenizer(
                    prompt_texts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=64,
                    add_special_tokens=False
                ).to(device)
        
                # ==========================================
                # Embedding
                # ==========================================
                with torch.no_grad():
        
                    token_embeds = model.embed(enc["input_ids"])
                
                    prompt = model.prompt_proj(latent)
    
                    prompt = prompt.view(latent.size(0), model.num_prefix, model.gpt2.config.n_embd)
                
                    inputs_embeds = torch.cat([prompt, token_embeds], dim=1)
    
                    prompt_mask = torch.ones(latent.size(0), model.num_prefix, device=device)
    
                    attention_mask = torch.cat([prompt_mask, enc["attention_mask"]], dim=1)
        
                    # ==========================================
                    # generate（100 Q&A Pairs）
                    # ==========================================
                    generated = model.gpt2.generate(
                        inputs_embeds=inputs_embeds,
                        attention_mask=attention_mask,
                        max_new_tokens=80,
                        do_sample=True,
                        temperature=0.7,
                        top_p=0.9,
                        eos_token_id=tokenizer.eos_token_id,
                        pad_token_id=tokenizer.pad_token_id,
                    )
        
                # ==========================================
                # decode（100 Q&A Pairs）
                # ==========================================
                decoded_list = tokenizer.batch_decode(generated, skip_special_tokens=True)
        
                result_true = []
                result_pred = []
        
                for answer, decoded in zip(answers, decoded_list):
                    result_true.append(extract_material_properties(answer))
                    result_pred.append(extract_material_properties(decoded))
        
                result_true_all.append(result_true)
                result_pred_all.append(result_pred)
        
            with open(path+"/result/"+result_path+"_true.json", "w", encoding="utf-8") as f:
                json.dump(result_true_all, f, ensure_ascii=False, indent=2)
        
            with open(path+"/result/"+result_path+"_pred.json", "w", encoding="utf-8") as f:
                json.dump(result_pred_all, f, ensure_ascii=False, indent=2)    

# Reproducibility notes (added)

Changes relative to the original notebook, made so that it runs outside a
batch-queue environment:

1. `seed = int(args[3])` and `latent_model = str(args[1])` were replaced by
   direct assignments. In the original, `args` is **never assigned** (there is no
   `args = sys.argv`), so both cells raised `NameError`.
2. `os.makedirs` now creates `model/Analysis7/<method>/` and
   `result/Analysis7/<method>/` before use. The original never created them, so
   a 500-epoch run ended in a failing `torch.save`.
3. The leading `/` was removed from `model_path` / `result_path`, which
   previously produced paths like `./model//Analysis7/...`.
4. `&` between two `os.path.exists` calls was changed to `and`.
5. The dead `if y is not None:` block inside `MultiTaskGPT2.forward` was removed.
   It ran a full `generate()` call and then discarded the result (`decoded` was
   assigned and immediately overwritten with `None`), and the `y` argument was
   never passed by the training loop.
6. Japanese comments were translated to English.

Not changed (deliberate, to keep the method identical): the prompt format
`Question:\n{q}\n\nAnswer:\n`, the frozen GPT-2 backbone, `num_prefix = 8`,
the projector shape (`latent_dim -> 512 -> num_prefix * n_embd`), the
block-mean normalisation of the embeddings, the `(epoch * 1000 + idx)` rule for
choosing the Q&A pair, and the sampling parameters
(`temperature=0.7`, `top_p=0.9`, `max_new_tokens=80`).

Known gaps that this notebook still has (see the repository's `src/` and the
review document):

- **No multi-task head.** The class is named `MultiTaskGPT2` and stores
  `alpha = 0.1`, but only the language-modelling loss is computed. The property
  values enter the pipeline exclusively through the generated text, parsed by
  regex. If the paper describes a multi-task objective, that code is missing.
- **No metrics and no plotting.** `r2_score` is imported but never called; the
  notebook stops after writing `*_true.json` / `*_pred.json`.
- **`y_train` / `y_test` are loaded and de-standardised but never used.** The
  de-standardisation also uses full-dataset statistics (a leak), which is
  harmless only while `y` is unused.
- **Evaluation is not deterministic.** `do_sample=True` with the torch RNG never
  seeded, so the figure's numbers cannot be reproduced.
- **The ground truth is parsed from text, not read from the measurement table.**
  A regex failure silently drops that property from the comparison, with no
  warning and no count.
- **`data/Q&A/treated/` is not produced by any supplied notebook.** Fig. 6 writes
  `raw_data/`; the filtering step that produces `treated/` is missing.
